# Global INT8 Quantize + Transpose

Bot task id: `13_quantize_global_transpose_int8`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/13_quantize_global_transpose_int8.ipynb)

## Условие

Глобальная симметричная квантизация в int8 с транспонированием. Дан 2D тензор `X` формы `(M, N)` (fp16/fp32, row-major, contiguous):
1. $\text{absmax} = \max |X|$ (скаляр fp32);
2. $Q = \operatorname{round}(127 \cdot X / \text{denom})$, где $\text{denom} = \max(\text{absmax},\ 127 \cdot 10^{-8})$;
3. результат пишется в `B` формы `(N, M)` int8, то есть сразу $Q^T$;
4. вернуть `(B, absmax[1])` — именно `absmax` без клэмпа; `denom` нужен только чтобы не делить на 0.

**Требования**
- одно Triton-ядро: грузит тайлы из `A` (`X`), квантизует и пишет в `B`, транспонируя индексы;
- `tl.load` / `tl.store` и маски по краям;
- `@triton.autotune` с не менее чем 3 конфигурациями (`BLOCK_M/N`, `GROUP_M`, `num_warps/num_stages`);
- произвольные совместимые страйды: у входа и у выхода хотя бы одно измерение unit-stride;
- обёртка `quantize_global_transpose(x)` возвращает `(q_T: int8[N, M], absmax: fp32[1])`.

Проверка включает вход из одних нулей.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `13_quantize_global_transpose_int8.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 13_quantize_global_transpose_int8.py
import torch
import triton
import triton.language as tl


@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 128, "GROUP_M": 8}, num_warps=4),
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 128, "GROUP_M": 8}, num_warps=4, num_stages=2),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64, "GROUP_M": 8}, num_warps=4, num_stages=2),
    ],
    key=["M", "N"],
)
@triton.jit
def _quantize_global_transpose(A, absmax_inv_ptr, B,
                               stride_am, stride_an,
                               stride_bn, stride_bm,
                               M, N,
                               BLOCK_M: tl.constexpr,
                               BLOCK_N: tl.constexpr,
                               GROUP_M: tl.constexpr):
    # YOUR CODE HERE
    pass


def quantize_global_transpose(x: torch.Tensor):
    """Return (q_T:int8[N,M], absmax:fp32[1])."""
    # YOUR CODE HERE
    raise NotImplementedError

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "13_quantize_global_transpose_int8.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

def quant_ref(x, denom):
    return torch.round(127.0 * x.float() / denom).clamp(-127, 127).to(torch.int8)


def assert_quant(q, ref):
    # int8 values may differ by 1 where rounding modes disagree on .5
    assert q.dtype == torch.int8, q.dtype
    assert q.shape == ref.shape, (q.shape, ref.shape)
    diff = (q.int() - ref.int()).abs()
    assert diff.max() <= 1, f"max |q - ref| = {diff.max().item()}"
    assert (diff > 0).float().mean() < 1e-3, f"{(diff > 0).sum().item()} elements off by 1"

for dtype in (torch.float32, torch.float16):
    x = torch.randn(300, 130, device="cuda", dtype=dtype)
    q_t, absmax = sol.quantize_global_transpose(x)
    ref_max = x.abs().max().float()
    assert absmax.shape == (1,) and absmax.dtype == torch.float32, (absmax.shape, absmax.dtype)
    torch.testing.assert_close(absmax[0], ref_max)
    assert_quant(q_t, quant_ref(x, ref_max).T.contiguous())
q_t, absmax = sol.quantize_global_transpose(torch.zeros(5, 3, device="cuda"))
assert absmax.item() == 0 and (q_t == 0).all(), "all-zero input must not divide by zero"
assert len(sol._quantize_global_transpose.configs) >= 3, "the task asks for at least 3 autotune configs"

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("13_quantize_global_transpose_int8.py")